In [1]:
import os

import pandas as pd
import numpy as np
import pickle

#NETTOYAGE
import re
import string
import nltk
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification

from sklearn.model_selection import train_test_split

## CONSTANTES

In [2]:
MAX_WORDS = 10000
MAX_LEN = 100
EMBEDDING_DIM = 64
EPOCHS = 3
BATCH_SIZE = 32

MODEL_NAME = "modele_bert"
TOKEN_NAME = "tokenizer.pickle"
OUTPUT_DIR = "../BERT"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

## Importation

In [3]:
columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv("../../../Data/training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)

## Nettoyage

In [4]:
#Fonction
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""

    #Passer en minuscule tout le texte
    texte = texte.lower()

    #Supprimer des éléments frequents dans des tweets, mais jugés ininteressants pour l'analyse
    texte = re.sub(r'<.*?>', '', texte)
    texte = re.sub(r'@\w+', '', texte)
    texte = re.sub(r'\d+', '', texte)
    texte = re.sub(r'https?://\S+|www\.\S+', '', texte)

    #Renvoie le texte nettoyé
    return texte


In [5]:
#Application
df_trt = df.copy()[['TARGET', 'id', 'date','user', 'tweet']]

df_trt['tweet_net'] = df_trt['tweet'].apply(nettoyer_texte)

## Tokenisation

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df_trt['tweet_net'][:2000], df_trt['TARGET'][:2000], test_size=0.2, random_state=42)

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def encode_texts(tokenizer, texts, max_leng):
    """Transforme les textes en format compréhensible par BERT"""
    texts = texts.tolist()
    texts_clean = [str(t) for t in texts]
    
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_leng,
        return_tensors="tf" # Format TensorFlow
    )

print("Encodage des données...")
train_encodings = encode_texts(tokenizer, X_train, MAX_LEN)
test_encodings = encode_texts(tokenizer, X_test, MAX_LEN)

# Conversion en TF Dataset (Optimisé pour Keras)
# BERT a besoin de 2 entrées : input_ids et attention_mask
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings), # Inputs (Dictionnaire)
    y_train                # Labels
)).shuffle(100).batch(BATCH_SIZE)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test
)).batch(BATCH_SIZE)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Encodage des données...


## BERT

### Construction du modèle

In [7]:
model = TFDistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=1,  # 1 seule sortie (Score entre 0 et 1)
    use_safetensors=False
)

# Compilation
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
# Important : from_logits=True car BERT sort un score brut non normalisé
loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_projector', 'vocab_layer_norm', 'activation_13', 'vocab_transform']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-

NameError: name 'LEARNING_RATE' is not defined

### Entrainement du modèle

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS
)

### Sauvegarde du modèle

In [ ]:
#TOKEN
tokenizer_path = os.path.join(OUTPUT_DIR, TOKEN_NAME)
with open(tokenizer_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

#MODELE
model_path = os.path.join(OUTPUT_DIR, MODEL_NAME)
model.save(model_path, save_format="tf")